# Writing to Delta Lake

Delta is a transactional table format on top of Parquet — it adds ACID, time travel, MERGE/UPDATE/DELETE, and schema evolution.

**Compatibility note**: this image runs Spark **4.1.4 / Scala 2.13**, so we use `delta-spark_2.13:4.3.1`. Picking the wrong Delta version for your Spark version is the #1 cause of cryptic class-not-found errors.

The first run downloads the jar via `spark.jars.packages` — that takes a moment. Subsequent runs use the cached jar.

In [ ]:
# Optional: install the Python API (DeltaTable). Skip if you only use SQL.
# ! pip install delta-spark==4.3.1

In [ ]:
from pyspark.sql import SparkSession, functions as F

In [ ]:
spark = (
    SparkSession.builder
    .appName("Write Delta")
    .master("local[2]")
    .config("spark.jars.packages", "io.delta:delta-spark_2.13:4.3.1")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

In [ ]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("file:///opt/examples/datasets/Mall_Customers.csv")

df.printSchema()
df.limit(5).toPandas()

## 1. Write — basic

In [ ]:
delta_path = "file:///tmp/pyspark_output_data/mall_delta"

df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(delta_path)

In [ ]:
# Notice the _delta_log/ folder — that's the transaction log
! ls -lh /tmp/pyspark_output_data/mall_delta
! ls -lh /tmp/pyspark_output_data/mall_delta/_delta_log

## 2. Read it back

In [ ]:
df_read = spark.read.format("delta").load(delta_path)
df_read.count()

## 3. Partitioned write

In [ ]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("Gender") \
    .save("file:///tmp/pyspark_output_data/mall_delta_partitioned")

In [ ]:
! ls /tmp/pyspark_output_data/mall_delta_partitioned

## 4. Append

In [ ]:
# Take 5 rows and append them to simulate new data
df_more = df.limit(5)
df_more.write.format("delta").mode("append").save(delta_path)

spark.read.format("delta").load(delta_path).count()

## 5. Time travel

Every write is a numbered version. You can read any past version.

In [ ]:
# version 0 = first overwrite, version 1 = the append above
spark.read.format("delta").option("versionAsOf", 0).load(delta_path).count()

In [ ]:
spark.read.format("delta").option("versionAsOf", 1).load(delta_path).count()

## 6. Update / Delete via SQL

In [ ]:
spark.sql(f"""
    UPDATE delta.`{delta_path}`
    SET SpendingScore = SpendingScore + 1
    WHERE Gender = 'Female'
""")

In [ ]:
spark.sql(f"""
    DELETE FROM delta.`{delta_path}`
    WHERE Age > 65
""")

## 7. History

In [ ]:
spark.sql(f"DESCRIBE HISTORY delta.`{delta_path}`").show(truncate=False)

In [ ]:
spark.stop()